### **Del alineamiento imagen-texto al alineamiento video-texto temporal**

Este cuaderno estudia la transición desde el alineamiento multimodal estático hacia el problema más exigente del **alineamiento video-texto**. A diferencia del caso imagen-texto, donde la correspondencia suele formularse a nivel global, en video surge una dimensión adicional: la **estructura temporal**. En consecuencia, ya no basta con identificar qué video se corresponde con una descripción, sino también **qué segmento o intervalo temporal del video** realiza el contenido semántico expresado por el texto.

#### **Eje conceptual**

El problema central consiste en pasar de un esquema de **alineamiento global video-texto** a mecanismos capaces de capturar correspondencias **más finas, localizadas y temporalmente sensibles**. Este cambio implica modelar no solo similitud semántica entre modalidades, sino también la manera en que dicha similitud se distribuye a lo largo de la secuencia visual.

Desde esta perspectiva, el cuaderno aborda dos ideas complementarias:

- el aprendizaje de representaciones compartidas entre video y texto mediante objetivos contrastivos,
- la incorporación de señales más estructuradas que permitan refinar el alineamiento sobre **segmentos, eventos o regiones temporales**.

#### **Modelos de referencia**

**1. VideoCLIP-lite**

**VideoCLIP-lite** representa una formulación contrastiva del alineamiento video-texto en la que la temporalidad cumple un papel explícito. El modelo aprende a asociar fragmentos de video y texto a partir de **pares positivos con solapamiento temporal**, mientras que mejora su capacidad discriminativa mediante **negativos difíciles** obtenidos con una cola de memoria.

La intuición es que el aprendizaje no debe limitarse a distinguir entre videos distintos, sino a reconocer cuáles fragmentos son semánticamente pertinentes dentro de un continuo temporal. Así, el alineamiento deja de ser puramente global y comienza a incorporar estructura temporal como parte del criterio de correspondencia.

**2. ALPRO-lite**

**ALPRO-lite** parte también de una base de alineamiento video-texto, pero amplía el entrenamiento con objetivos complementarios que refinan la relación entre ambas modalidades:

- **VTC (Video-Text Contrastive):** favorece la cercanía entre representaciones de video y texto semánticamente consistentes,
- **VTM (Video-Text Matching):** introduce una señal binaria de correspondencia que obliga al modelo a distinguir pares alineados de pares no alineados,
- **PEM-lite:** incorpora una forma ligera de *prompting* orientada a **entidades o segmentos**, operando sobre pseudo-regiones temporales para inyectar estructura adicional en la asociación entre texto y contenido visual.

En conjunto, esta estrategia no solo fortalece el alineamiento global, sino que también promueve una interacción más localizada entre descripciones lingüísticas y subestructuras temporales del video.

##### **Pregunta guía**

> ¿Qué capacidades adicionales adquiere un modelo cuando pasa de un alineamiento global video-texto a un esquema que incorpora señales más finas sobre segmentos o regiones temporales?


In [ ]:
# 0. Setup

import math
import random
import time
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Dict, List, Optional, Any

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

def resolve_device(preference="auto"):
    preference = preference.lower().strip()
    if preference == "auto":
        return torch.device("cuda" if torch.cuda.is_available() else "cpu")
    if preference == "cuda":
        if torch.cuda.is_available():
            return torch.device("cuda")
        print("CUDA no disponible, usando CPU.")
        return torch.device("cpu")
    return torch.device("cpu")

print("Torch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

#### **1. Configuración**

Por defecto, el cuaderno emplea un **dataset sintético con estructura temporal**, de modo que pueda ejecutarse de manera reproducible en prácticamente cualquier entorno.

No obstante, se dejan preparados los puntos de extensión necesarios para adaptarlo posteriormente a datasets reales, tales como:

- **MSRVTT-like**
- **DiDeMo-like**
- **WebVid-like**

Desde el punto de vista metodológico, el interés principal del cuaderno no reside en el dataset específico, sino en la lógica experimental que organiza el problema. En particular, se enfatiza:

- la construcción de pares **video-texto**,
- la definición de **positivos temporales**,
- la incorporación de **negativos difíciles**,
- la comparación entre un esquema de **alineamiento global** y otro que combina **alineamiento global** con una **señal fina de localización temporal**.

In [ ]:
# 1. Configuración

@dataclass
class Config:
    device_preference: str = "auto"
    batch_size: int = 8
    num_workers: int = 0
    use_amp: bool = True

    vocab_size: int = 2048
    d_model: int = 128
    num_heads: int = 4
    ff_mult: int = 4
    dropout: float = 0.1

    max_text_len: int = 20
    max_video_len: int = 24
    num_segments: int = 4

    train_size: int = 256
    val_size: int = 64
    test_size: int = 64

    epochs_videoclip: int = 2
    epochs_alpro: int = 2

    lr: float = 1e-3
    weight_decay: float = 1e-4
    memory_bank_size: int = 256
    hard_negative_k: int = 4

    num_entities: int = 12

CFG = Config()
DEVICE = resolve_device(CFG.device_preference)
USE_AMP = bool(CFG.use_amp and DEVICE.type == "cuda")
CFG

#### **2. Dataset sintético temporal**

Cada muestra del dataset está compuesta por una secuencia de video, un texto asociado, un intervalo temporal activo y un conjunto de entidades dominantes.

El objetivo de este diseño no es reproducir con fidelidad el realismo visual de un dataset natural, sino **modelar de manera controlada la estructura esencial del problema**. En particular, se busca simular una situación en la que el texto se alinea con un fragmento específico del video y no necesariamente con su totalidad. De este modo, el dataset permite distinguir entre señales de carácter global, que describen propiedades generales de la secuencia, y señales locales o temporales, que solo resultan informativas en determinados segmentos.

In [ ]:
# 2. Dataset sintético temporal

ENTITY_VOCAB = [
    "person", "dog", "cat", "car", "tree", "ball",
    "music", "speech", "rain", "wind", "water", "bird"
]

def lengths_to_mask(lengths: torch.Tensor, max_len: int) -> torch.Tensor:
    ar = torch.arange(max_len).unsqueeze(0)
    return ar < lengths.unsqueeze(1)

class SyntheticTemporalVideoTextDataset(Dataset):
    def __init__(self, size: int, cfg: Config, split: str = "train"):
        self.size = size
        self.cfg = cfg
        self.split = split
        self.samples = [self._make_sample(i) for i in range(size)]

    def _make_sample(self, idx: int):
        L = random.randint(max(8, self.cfg.max_video_len // 2), self.cfg.max_video_len)
        T = random.randint(max(6, self.cfg.max_text_len // 2), self.cfg.max_text_len)

        video = torch.randn(L, self.cfg.d_model) * 0.2
        text_tokens = torch.randint(0, self.cfg.vocab_size, (T,), dtype=torch.long)

        # segmento activo temporal
        seg_len = max(3, L // self.cfg.num_segments)
        start = random.randint(0, max(0, L - seg_len))
        end = min(L, start + seg_len)

        # dos entidades dominantes
        e1 = random.randint(0, len(ENTITY_VOCAB) - 1)
        e2 = random.randint(0, len(ENTITY_VOCAB) - 1)

        # sesgo estructural para que el modelo pueda aprender algo
        video[start:end] += 0.8
        if T >= 2:
            text_tokens[0] = e1 + 10
            text_tokens[1] = e2 + 10

        return {
            "id": f"{self.split}_{idx}",
            "video": video,
            "video_length": L,
            "text_tokens": text_tokens,
            "text_length": T,
            "active_start": start,
            "active_end": end,
            "entities": torch.tensor([e1, e2], dtype=torch.long),
        }

    def __len__(self):
        return self.size

    def __getitem__(self, idx):
        return self.samples[idx]

def collate_temporal(batch: List[Dict[str, Any]], cfg: Config):
    B = len(batch)
    Lmax = cfg.max_video_len
    Tmax = cfg.max_text_len
    D = cfg.d_model

    video = torch.zeros(B, Lmax, D)
    text_tokens = torch.zeros(B, Tmax, dtype=torch.long)
    video_lengths = torch.zeros(B, dtype=torch.long)
    text_lengths = torch.zeros(B, dtype=torch.long)
    active_start = torch.zeros(B, dtype=torch.long)
    active_end = torch.zeros(B, dtype=torch.long)
    entities = torch.zeros(B, 2, dtype=torch.long)
    ids = []

    for i, item in enumerate(batch):
        Lv = min(item["video"].shape[0], Lmax)
        Lt = min(item["text_tokens"].shape[0], Tmax)
        video[i, :Lv] = item["video"][:Lv]
        text_tokens[i, :Lt] = item["text_tokens"][:Lt]
        video_lengths[i] = Lv
        text_lengths[i] = Lt
        active_start[i] = min(int(item["active_start"]), Lmax - 1)
        active_end[i] = min(int(item["active_end"]), Lmax)
        entities[i] = item["entities"]
        ids.append(item["id"])

    return {
        "ids": ids,
        "video": video,
        "text_tokens": text_tokens,
        "video_lengths": video_lengths,
        "text_lengths": text_lengths,
        "video_mask": lengths_to_mask(video_lengths, Lmax),
        "text_mask": lengths_to_mask(text_lengths, Tmax),
        "active_start": active_start,
        "active_end": active_end,
        "entities": entities,
    }

train_ds = SyntheticTemporalVideoTextDataset(CFG.train_size, CFG, "train")
val_ds = SyntheticTemporalVideoTextDataset(CFG.val_size, CFG, "val")
test_ds = SyntheticTemporalVideoTextDataset(CFG.test_size, CFG, "test")

train_loader = DataLoader(train_ds, batch_size=CFG.batch_size, shuffle=True, num_workers=CFG.num_workers, collate_fn=lambda b: collate_temporal(b, CFG))
val_loader = DataLoader(val_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=lambda b: collate_temporal(b, CFG))
test_loader = DataLoader(test_ds, batch_size=CFG.batch_size, shuffle=False, num_workers=CFG.num_workers, collate_fn=lambda b: collate_temporal(b, CFG))

batch0 = next(iter(train_loader))
{k: tuple(v.shape) if hasattr(v, "shape") else type(v) for k, v in batch0.items()}

#### **3. Visualización conceptual**

La noción de **positivo con solapamiento temporal** puede entenderse del siguiente modo: el texto asociado no describe necesariamente la totalidad del video, sino que encuentra su mejor correspondencia en un **subtramo específico** de la secuencia.

En el dataset sintético, dicho subtramo se encuentra definido explícitamente por las variables `active_start` y `active_end`, que delimitan el intervalo temporal en el que la correspondencia video-texto resulta más fuerte.

In [ ]:
sample = train_ds[0]
L = sample["video_length"]
start = sample["active_start"]
end = sample["active_end"]

timeline = np.zeros(L)
timeline[start:end] = 1

plt.figure(figsize=(10, 2))
plt.plot(timeline, marker="o")
plt.ylim(-0.1, 1.1)
plt.title(f"Segmento activo de la muestra {sample['id']}")
plt.xlabel("tiempo")
plt.ylabel("activo")
plt.show()

print("Tokens de texto:", sample["text_tokens"][:8].tolist())
print("Entidades:", [ENTITY_VOCAB[i] for i in sample["entities"].tolist()])

#### **4. Bloques base**

In [ ]:
# 4. Bloques base

class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model: int, max_len: int = 512):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class FeedForward(nn.Module):
    def __init__(self, d_model: int, mult: int = 4, dropout: float = 0.1):
        super().__init__()
        hidden = d_model * mult
        self.net = nn.Sequential(
            nn.Linear(d_model, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, d_model),
        )
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int, dropout: float):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads, dropout=dropout, batch_first=True)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = FeedForward(d_model, ff_mult, dropout)
        self.drop = nn.Dropout(dropout)

    def forward(self, x, key_padding_mask=None):
        h = self.ln1(x)
        out, _ = self.attn(h, h, h, key_padding_mask=(~key_padding_mask if key_padding_mask is not None else None))
        x = x + self.drop(out)
        x = x + self.drop(self.ff(self.ln2(x)))
        return x

def mean_pool_masked(x, mask):
    m = mask.float().unsqueeze(-1)
    return (x * m).sum(dim=1) / m.sum(dim=1).clamp_min(1.0)

class TinyTextEncoder(nn.Module):
    def __init__(self, vocab_size: int, d_model: int, num_heads: int, ff_mult: int, dropout: float):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, 512)
        self.block = TransformerBlock(d_model, num_heads, ff_mult, dropout)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, tokens, mask):
        x = self.embed(tokens)
        x = self.pos(x)
        x = self.block(x, mask)
        return F.normalize(self.proj(mean_pool_masked(x, mask)), dim=-1)

class TinyVideoEncoder(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int, dropout: float):
        super().__init__()
        self.pos = SinusoidalPositionalEncoding(d_model, 512)
        self.block = TransformerBlock(d_model, num_heads, ff_mult, dropout)
        self.proj = nn.Linear(d_model, d_model)

    def forward(self, video, mask):
        x = self.pos(video)
        x = self.block(x, mask)
        return F.normalize(self.proj(mean_pool_masked(x, mask)), dim=-1)

#### **5. VideoCLIP-lite**

Este modelo parte de una formulación de **alineamiento contrastivo global** entre video y texto, reforzada mediante un mecanismo de **negativos difíciles** basado en una memoria de corto alcance.

En esta implementación no se recurre a una *recuperación* mediante ANN real. En su lugar, se emplea una **cola de embeddings**, que constituye una aproximación computacionalmente económica y suficientemente expresiva para los fines del cuaderno.

In [ ]:
# 5. VideoCLIP-lite

class MemoryQueue(nn.Module):
    def __init__(self, d_model: int, max_size: int):
        super().__init__()
        self.max_size = max_size
        self.register_buffer("queue", torch.empty(0, d_model), persistent=False)

    @torch.no_grad()
    def enqueue(self, x):
        x = x.detach()
        if self.queue.numel() == 0:
            self.queue = x[-self.max_size:]
        else:
            self.queue = torch.cat([self.queue, x], dim=0)[-self.max_size:]

    def get(self):
        return self.queue

class VideoCLIPLite(nn.Module):
    def __init__(self, cfg: Config):
        super().__init__()
        self.video_enc = TinyVideoEncoder(cfg.d_model, cfg.num_heads, cfg.ff_mult, cfg.dropout)
        self.text_enc = TinyTextEncoder(cfg.vocab_size, cfg.d_model, cfg.num_heads, cfg.ff_mult, cfg.dropout)
        self.temp = 0.07
        self.k = cfg.hard_negative_k
        self.mem_v = MemoryQueue(cfg.d_model, cfg.memory_bank_size)
        self.mem_t = MemoryQueue(cfg.d_model, cfg.memory_bank_size)

    def forward(self, batch):
        zv = self.video_enc(batch["video"], batch["video_mask"])
        zt = self.text_enc(batch["text_tokens"], batch["text_mask"])

        logits = zv @ zt.T / self.temp
        labels = torch.arange(zv.size(0), device=zv.device)
        loss_nce = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

        loss_hard = torch.tensor(0.0, device=zv.device)
        qv = self.mem_v.get()
        qt = self.mem_t.get()

        if qt.numel() > 0:
            sim = zv @ qt.T
            hard = sim.topk(k=min(self.k, sim.size(1)), dim=1).values
            pos = (zv * zt).sum(dim=-1, keepdim=True)
            logits_h = torch.cat([pos, hard], dim=1) / self.temp
            loss_hard = loss_hard + F.cross_entropy(logits_h, torch.zeros(zv.size(0), dtype=torch.long, device=zv.device))

        if qv.numel() > 0:
            sim = zt @ qv.T
            hard = sim.topk(k=min(self.k, sim.size(1)), dim=1).values
            pos = (zt * zv).sum(dim=-1, keepdim=True)
            logits_h = torch.cat([pos, hard], dim=1) / self.temp
            loss_hard = loss_hard + F.cross_entropy(logits_h, torch.zeros(zt.size(0), dtype=torch.long, device=zt.device))

        with torch.no_grad():
            self.mem_v.enqueue(zv)
            self.mem_t.enqueue(zt)

        return {
            "loss": loss_nce + 0.25 * loss_hard,
            "retrieval_logits": logits,
            "zv": zv,
            "zt": zt,
        }

#### **6. ALPRO-lite temporal**

En esta variante se incorporan tres señales de entrenamiento complementarias:

- **VTC** (*Video-Text Contrastive*): favorece el alineamiento global entre video y texto,
- **VTM** (*Video-Text Matching*): introduce una señal binaria de correspondencia entre ambas modalidades,
- **PEM-lite**: añade un mecanismo de *prompting* aplicado sobre un segmento pseudo-activo.

En la implementación del notebook, los segmentos activos sintéticos se utilizan como una aproximación computacionalmente económica a la noción de **región o segmento temporal relevante**, permitiendo capturar de forma simple una señal más fina que el alineamiento estrictamente global.

In [ ]:
# 6. ALPRO-lite temporal

class MultiModalFusion(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int, dropout: float):
        super().__init__()
        self.block = TransformerBlock(d_model, num_heads, ff_mult, dropout)

    def forward(self, v_tokens, t_tokens, v_mask=None, t_mask=None):
        x = torch.cat([v_tokens, t_tokens], dim=1)
        mask = torch.cat([v_mask, t_mask], dim=1) if (v_mask is not None and t_mask is not None) else None
        return self.block(x, mask)

class ALPROLiteTemporal(nn.Module):
    def __init__(self, cfg: Config, entity_vocab: List[str]):
        super().__init__()
        self.entity_vocab = entity_vocab
        self.text_global = TinyTextEncoder(cfg.vocab_size, cfg.d_model, cfg.num_heads, cfg.ff_mult, cfg.dropout)
        self.video_global = TinyVideoEncoder(cfg.d_model, cfg.num_heads, cfg.ff_mult, cfg.dropout)
        self.text_embed = nn.Embedding(cfg.vocab_size, cfg.d_model)
        self.mm = MultiModalFusion(cfg.d_model, cfg.num_heads, cfg.ff_mult, cfg.dropout)
        self.vtm_head = nn.Linear(cfg.d_model, 2)
        self.pem_head = nn.Linear(cfg.d_model, len(entity_vocab))

    def build_prompt_bank(self, device):
        bank = torch.randn(len(self.entity_vocab), CFG.d_model, device=device)
        return F.normalize(bank, dim=-1)

    def forward(self, batch):
        # VTC
        zv = self.video_global(batch["video"], batch["video_mask"])
        zt = self.text_global(batch["text_tokens"], batch["text_mask"])
        logits = zv @ zt.T / 0.07
        labels = torch.arange(zv.size(0), device=zv.device)
        vtc_loss = 0.5 * (F.cross_entropy(logits, labels) + F.cross_entropy(logits.T, labels))

        # Fusión multimodal
        v_tokens = batch["video"]
        t_tokens = self.text_embed(batch["text_tokens"])
        fused = self.mm(v_tokens, t_tokens, batch["video_mask"], batch["text_mask"])
        cls = fused[:, 0]

        # VTM
        vtm_logits = self.vtm_head(cls)
        vtm_labels = torch.ones(vtm_logits.size(0), dtype=torch.long, device=cls.device)
        vtm_loss = F.cross_entropy(vtm_logits, vtm_labels)

        # PEM-lite: usa el segmento activo sintético
        seg_feats = []
        for i in range(batch["video"].size(0)):
            s = int(batch["active_start"][i].item())
            e = int(batch["active_end"][i].item())
            seg = batch["video"][i, s:e]
            seg_feats.append(seg.mean(dim=0))
        seg_feats = torch.stack(seg_feats, dim=0)

        prompt_bank = self.build_prompt_bank(cls.device).detach()
        q = F.softmax(F.normalize(seg_feats, dim=-1) @ prompt_bank.T, dim=-1)
        p = F.log_softmax(self.pem_head(seg_feats), dim=-1)
        pem_loss = -(q * p).sum(dim=-1).mean()

        total = vtc_loss + vtm_loss + pem_loss
        return {
            "loss": total,
            "retrieval_logits": logits,
            "vtc_loss": vtc_loss.detach(),
            "vtm_loss": vtm_loss.detach(),
            "pem_loss": pem_loss.detach(),
            "zv": zv,
            "zt": zt,
        }

#### **7. Entrenamiento y evaluación**

In [ ]:
def move_batch_to_device(batch, device):
    return {k: (v.to(device) if torch.is_tensor(v) else v) for k, v in batch.items()}

@torch.no_grad()
def recall_at_k(sim, k=1):
    topk = sim.topk(min(k, sim.size(1)), dim=1).indices
    labels = torch.arange(sim.size(0), device=sim.device).unsqueeze(1)
    return float((topk == labels).any(dim=1).float().mean().cpu())

@torch.no_grad()
def retrieval_metrics(sim):
    return {
        "R@1": recall_at_k(sim, 1),
        "R@5": recall_at_k(sim, 5),
        "R@10": recall_at_k(sim, 10),
    }

def train_one_epoch(model, loader, optimizer, device):
    model.train()
    losses = []
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        optimizer.zero_grad()
        out = model(batch)
        loss = out["loss"]
        loss.backward()
        optimizer.step()
        losses.append(float(loss.detach().cpu()))
    return float(np.mean(losses))

@torch.no_grad()
def evaluate_retrieval(model, loader, device):
    model.eval()
    sims = []
    for batch in loader:
        batch = move_batch_to_device(batch, device)
        out = model(batch)
        sims.append(out["retrieval_logits"].cpu())
    sim = torch.block_diag(*sims) if len(sims) > 1 else sims[0]
    return retrieval_metrics(sim)

#### **8. Experimento A: VideoCLIP-lite**

In [ ]:
videoclip_lite = VideoCLIPLite(CFG).to(DEVICE)
opt_vc = torch.optim.AdamW(videoclip_lite.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

vc_history = []
for epoch in range(CFG.epochs_videoclip):
    tr_loss = train_one_epoch(videoclip_lite, train_loader, opt_vc, DEVICE)
    val_scores = evaluate_retrieval(videoclip_lite, val_loader, DEVICE)
    row = {"epoch": epoch + 1, "train_loss": round(tr_loss, 4), **{k: round(v, 4) for k, v in val_scores.items()}}
    vc_history.append(row)
    print(row)

vc_df = pd.DataFrame(vc_history)
vc_df

#### **9. Experimento B: ALPRO-lite temporal**

In [ ]:
alpro_lite = ALPROLiteTemporal(CFG, ENTITY_VOCAB[:CFG.num_entities]).to(DEVICE)
opt_alpro = torch.optim.AdamW(alpro_lite.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)

alpro_history = []
for epoch in range(CFG.epochs_alpro):
    tr_loss = train_one_epoch(alpro_lite, train_loader, opt_alpro, DEVICE)
    val_scores = evaluate_retrieval(alpro_lite, val_loader, DEVICE)
    row = {"epoch": epoch + 1, "train_loss": round(tr_loss, 4), **{k: round(v, 4) for k, v in val_scores.items()}}
    alpro_history.append(row)
    print(row)

alpro_df = pd.DataFrame(alpro_history)
alpro_df

#### **10. Comparación de resultados**

In [ ]:
final_results = pd.DataFrame([
    {"modelo": "VideoCLIP-lite", **vc_history[-1]},
    {"modelo": "ALPRO-lite temporal", **alpro_history[-1]},
])
final_results

In [ ]:
plt.figure(figsize=(8, 4))
plt.plot(vc_df["epoch"], vc_df["R@1"], marker="o", label="VideoCLIP-lite R@1")
plt.plot(alpro_df["epoch"], alpro_df["R@1"], marker="o", label="ALPRO-lite R@1")
plt.xlabel("epoch")
plt.ylabel("R@1")
plt.title("Comparación de retrieval")
plt.legend()
plt.show()

#### **11. Preguntas de discusión**

1. ¿Qué parte del problema video-texto no existía en imagen-texto?
2. ¿Cuándo basta un bi-encoder y cuándo conviene añadir una señal tipo PEM?
4. ¿Este cuaderno prepara mejor el salto a papers de transformers crossmodales?
5. ¿Qué cambiarías primero para conectarlo a MSRVTT o WebVid reales?

#### **12. Trabajo futuro sugerido**

Este cuaderno puede extenderse mediante una serie de ejercicios orientados a profundizar tanto en la implementación como en el marco conceptual del alineamiento video-texto:

1. Conecta el pipeline a un dataset real, por ejemplo de tipo **MSRVTT**.
2. Reemplaza el dataset sintético por clips reales con anotaciones temporales (*timestamps*).
3. Añade un **cross-reranker temporal** para refinar el ranking inicial de candidatos.
4. Diseña una evaluación que considere al menos tres tareas:
   - recuperación **texto ->video**,
   - recuperación **video ->  texto**,
   - decisión binaria de correspondencia.
5. Emplear el cuaderno como un puente hacia el estudio más formal de **VideoCLIP** y **ALPRO**, y posteriormente hacia mecanismos de **atención crossmodal no alineada**.